In [1]:
import pandas as pd
import os
import numpy as np

In [6]:
input_file = 'msms.txt'
peptides_table_sbm3 = 'peptides.txt'
output_file_sbm3  = 'percolator_pin'

# Input Features to Percolator

From crux website: https://crux.ms/file-formats/features.html 

Field		Value
scan		10
charge		2
spectrum precursor m/z	636.3400
spectrum neutral mass	1270.6644
peptide mass		1270.4917
delta_cn		0
sp score		56.458275
sp rank		1
xcorr score		0.843481
xcorr rank		1
b/y ions matched	5
b/y ions total	22
matches/spectrum	6
sequence		SGLIVEIQGVQK
cleavage type		trypsin-full-digest
protein id		YDR093W
flanking aa		KE
unshuffled sequence	(empty)


#### converting mq output
SpecId  =  (Identifier of this peptide-spectrum match. If the PIN file was created by Crux, then the ID will be of the form target_0_8000_2_1, where the components are "target" or "decoy," the file index, scan number, charge, and PSM rank.) (103111-Yeast-2hr-01_27_2_1)
Label = 'Reverse' (shold be converted from NAN to -1 and + to -1 )
ScanNr = 'Scan number' 
ExpMass = 'm/z' (precursor m/z) 
CalcMass =  'Mass'
PSMScore = 'Score'
ChargeN = 'Charge'
Peptide	= 'Modified sequence'
Proteins = 'Proteins' (should be writrten seperated by commas)

#### example dataframe
SpecId	Label	ScanNr	ExpMass	CalcMass	lnrSp	deltLCn	deltCn	Xcorr	Sp	IonFrac	Mass	PepLen	Charge1	Charge2	Charge3	Charge4	Charge5	enzN	enzC	enzInt	lnNumSP	dM	absdM	Peptide	Proteins
103111-Yeast-2hr-01_27_2_1	1	27	1139.57	1139.57	0.693147	0.0121837	0.0121837	0.757094	98.633	0.375	1139.57	9	0	1	0	0	0	1	1	0	5.34711	0.00275	0.00275	R.LFLVM[16]DEEK.N	sp|P40497|YIJ2_YEAST


## digestion 

* Asp-N cleaves at the N-terminus of aspartic (D) and cysteic acid residues with high specificity (1–3).
* Endoproteinase Lys-C is a protease that cleaves proteins on the C-terminal side of lysine residues. (K) 
* glu -c 

### N term
- peptide should begin with K or previous peptide end with D or E

### C term 
-  Peptide should begin with D or E or end with K
-  if SUMO on end ,

percolator_df['enzN'] = true if 'Amino acid before' == Nan or 'K' or if 'First amino acid' = D or E 


percolator_df['enzC'] = true if 'Last amino acid' == 'K' or if 'Amino acid after' = D or E or nan 

In [4]:
def convert_maxquant_to_percolator(input_file, output_file, peptides_table, mod):
    
    # Load MaxQuant data
    df = pd.read_csv(input_file, sep='\t') 
    
    # read peptides table for digestion pattern
    df_peps = pd.read_csv(peptides_table, delimiter="\t")
    df_peps['Evidence ID'] = df_peps['Evidence IDs'].str.split(';') 
    df_peps = df_peps.explode('Evidence ID', ignore_index=True)
    df_peps['Evidence ID']  = df_peps['Evidence ID'].astype(int)

    df = pd.merge(df, df_peps, on=['Evidence ID', 'Sequence', 'Reverse'], how='inner', suffixes=('', '_peps'))
    
    percolator_df = pd.DataFrame()
    percolator_df['SpecId'] = df.apply(lambda x: f"{'decoy' if pd.notna(x['Reverse']) else 'target'}_0_{x['Scan number']}_{x['Raw file']}_{x['Charge']}_1", axis=1)
    percolator_df['Label'] = df['Reverse'].apply(lambda x: -1 if pd.notna(x) else 1) 
    percolator_df['ScanNr'] = df['Scan number']
    percolator_df['ExpMass'] = df['m/z']
    percolator_df['CalcMass'] = df['Mass']
    df['Rank'] = df['Score'].rank(method='dense', ascending=False)
    percolator_df['lnrSp'] = np.log(df['Rank']) 
    df['Last score'] = df['All scores'].str.split(';').str[-1].astype(float)
    percolator_df['deltLCn'] = (df['Score'] - df['Last score'] ) / np.maximum(df['Score'], 1)  # divided by this PSM's score or 1, whichever is larger.
    percolator_df['deltCn'] = (df['Delta score'] ) / np.maximum(df['Score'], 1)
    #percolator_df['Xcorr'] = df['Score']  
    percolator_df['Sp'] = df['Score'] 
    percolator_df['IonFrac'] = df['Peak coverage']
    percolator_df['Mass'] = df['Mass']  
    percolator_df['PepLen'] = df['Length']
    percolator_df['ChargeN'] = df['Charge']

    # if you want charge serpeated into different columns
    #unique_charges = sorted(df['Charge'].dropna().unique())
    #for charge in unique_charges:
    #    col_name = f'Charge{int(charge)}'
    #    percolator_df[col_name] = (df['Charge'] == charge).astype(int)
    
    percolator_df['enzInt'] = df['Missed cleavages']
    
    #percolator_df['dM'] = df['Mass error [Da]'].fillna(0) # not this , has missing values df['Simple mass error [ppm]']  #
    #percolator_df['absdM'] = df['Mass error [Da]'].abs().fillna(0)  #, has missing values 

    if mod == 'sumo':
        df['Modified sequence'] = df['Modified sequence'].str.replace(r'st_o_SUMO2/3-DVFQQQTGG|ST_2_SUMO2/3-DVFQQQTGG|SBM_4_o_SUMO2/3-DVFQQQTGG|SBM_5_o_SUMO2/3-DVFQQQTGG|SBM_3_o_SUMO2/3-DVFQQQTGG|SBM_2_o_SUMO2/3-DVFQQQTGG|SBM_8_o_SUMO2/3-DVFQQQTGG', 'su', regex=True)

        df['Modified sequence'] = df['Modified sequence'].str.replace('Acetyl (Protein N-term)', 'ac', regex=False).str.replace('Oxidation (M)', 'ox', regex=False)
    
        # String formatting peptides from () to [] notation
        df['Modified sequence'] = df['Modified sequence'].str.replace('(', '[', regex=False).str.replace(')', ']', regex=False).str.replace('_', '.', regex=False)
        df['has_su'] = df['Modified sequence'].str.endswith('[su].')

        # SUMO specific, change to your enzyme digestion pettern
        percolator_df['enzN'] = (
            ((df['Amino acid before'].isna()) | 
             (df['Amino acid before'] == 'K') | 
             (df['First amino acid'].isin(['D', 'E'])))
            .astype(int)
        )
        
        percolator_df['enzC'] = (
            (((df['Last amino acid'] == 'K') | 
              (df['Amino acid after'].isin(['D', 'E'])) | 
              (df['Amino acid after'].isna())) &  
             (~df['has_su']) |  
             ((df['has_su']) & (df['Last amino acid'] == 'K') & 
              (df['Amino acid after'].isin(['D', 'E']))))
            .astype(int)
        )

    elif mod == 'ubi':

        df['Modified sequence'] = df['Modified sequence'].str.replace(r'Ubi_3_o', 'ubi', regex=True)

        #Simplify modification names 
        df['Modified sequence'] = df['Modified sequence'].str.replace('Acetyl (Protein N-term)', 'ac', regex=False).str.replace('Oxidation (M)', 'ox', regex=False)
    
        # String formatting peptides from () to [] notation
        df['Modified sequence'] = df['Modified sequence'].str.replace('(', '[', regex=False).str.replace(')', ']', regex=False).str.replace('_', '.', regex=False)
        df['has_ubi'] = df['Modified sequence'].str.endswith('[ubi].')

        percolator_df['enzN'] = (
            ((df['Amino acid before'].isna()) | 
             (df['Amino acid before'] == 'K'))
            .astype(int)
        )
        
        percolator_df['enzC'] = (
            (((df['Last amino acid'] == 'K') | 
              (df['Amino acid after'].isna()))) #  &  (~df['has_ubi'])
            .astype(int)
        )
    elif mod == 'none':
        # use trypsin digestion pattern 
        None
    
    # Formatting Peptide sequences
    percolator_df['Peptide'] = df['Modified sequence'] 
    percolator_df['Proteins'] = df['Proteins'].apply(lambda x: ','.join(str(x).split(';')) if pd.notna(x) else 'unknown') 
    
    # Save to output file
    percolator_df.to_csv(output_file, sep='\t', index=False)
    print(f'Converted file saved to {output_file}')

    return percolator_df

In [ ]:
per_df = convert_maxquant_to_percolator(input_file, output_file, peptides_table, mod ='sumo')